In [ ]:
# this notebook conduct the following task:
# 1 the origianl non redundant dataset was downloaded from here http://www.abybank.org/abdb/
#   \My Drive\@antibody\antigen_epitope\abdb_v2\data\abdb_data

# 2 Extract interaction info (tsv files) according to minumn atom-atom distance between sidechains of complexes
#   These tsv files have no SI information
#   structures_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/structures/"
#   output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_info/"

# 3 Extract antigen's structure info and generate the mesh objs files using pymol
#   antigen_pdb_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_pdbs'
#   antigen_obj_dir = '/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_objs'

# 4 add surface index to the antigen pdb file
#   SI_output_dir = "/content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data/antigen_pdb_with_SI"

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
# first-time use execution
!pip install biopython

In [ ]:
!pip install scipy

In [ ]:
!pip install matplotlib ipywidgets

In [ ]:
!pip install pywavefront
!pip install ipyvolume
!pip install pyvista
!pip install matplotlib ipywidgets

In [ ]:
!pip install trimesh
!pip install rtree

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

Counting amount of files

In [ ]:
#@title Generate antigen pdb with SI

In [ ]:
import ipyvolume as ipv
from matplotlib import cm
import pyvista as pv
import pywavefront
import os

In [ ]:
# dir to save info with SI
SI_output_dir = "/content/drive/MyDrive/@antigen_epitope/abdb_combine/data/antigen_pdb_with_SI"
antigen_pdb_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_combine/data/antigen_pdbs'
antigen_obj_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_combine/data/antigen_objs'

In [ ]:
#@title Add Shape Index (yuhao method 4)
import numpy as np
import trimesh
import os
import pandas as pd
from Bio import PDB
from scipy.spatial import KDTree

first_obj = next(f for f in os.listdir(antigen_obj_dir) if f.endswith('.obj'))
mesh = trimesh.load_mesh(os.path.join(antigen_obj_dir, first_obj))
vertices = mesh.vertices
faces = mesh.faces

K = trimesh.curvature.discrete_gaussian_curvature_measure(mesh, mesh.vertices, 2)
H = trimesh.curvature.discrete_mean_curvature_measure(mesh, mesh.vertices, 2)
elem = np.square(H) - K
elem[elem <= 0] = 1e-8
k1 = H + np.sqrt(elem)
k2 = H - np.sqrt(elem)
shape_index = (2.0 / np.pi) * np.arctan((k1 + k2) / (k1 - k2))

obj_coords = mesh.vertices
kd_tree = KDTree(obj_coords)

for pdb_filename in os.listdir(antigen_pdb_dir):
    if not pdb_filename.endswith('.pdb'):
        continue

    pdb_filepath = os.path.join(antigen_pdb_dir, pdb_filename)

    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_filepath)

    pdb_coords = []
    for atom in structure.get_atoms():
        pdb_coords.append(atom.get_coord())
    pdb_coords = np.array(pdb_coords)

    distances, indices = kd_tree.query(pdb_coords)

    for atom_index, atom in enumerate(structure.get_atoms()):
        atom.set_bfactor(shape_index[indices[atom_index]])

    output_pdb_filename = os.path.splitext(pdb_filename)[0] + '_updated.pdb'
    output_pdb_filepath = os.path.join(SI_output_dir, output_pdb_filename)

    os.makedirs(SI_output_dir, exist_ok=True)

    io = PDB.PDBIO()
    io.set_structure(structure)
    io.save(output_pdb_filepath)
